In [1]:
#packages
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
import math as m



In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/09 19:52:56 WARN Utils: Your hostname, Dions-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.86 instead (on interface en0)
25/10/09 19:52:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/09 19:52:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/09 19:52:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 65501)
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/Users/dionpapadopoulos/Library/Python/3.9/li

In [3]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    
    # === MEMORY MANAGEMENT ===
    .config("spark.driver.memory", "4g")          # Increase driver memory (safe for 16GB+ systems)
    .config("spark.executor.memory", "4g")        # Executors share same JVM locally
    .config("spark.driver.maxResultSize", "2g")   # Prevent large collect() results crashing driver

    # === PARALLELISM & SHUFFLING ===
    .config("spark.sql.shuffle.partitions", "48")  # Default is 200 — too high locally
    .config("spark.default.parallelism", "8")      # ~ number of cores on your system
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # Optimal shuffle partition size

    # === PERFORMANCE TUNING ===
    .config("spark.memory.fraction", "0.85")        # 85% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.3")  # 30% of execution memory for caching
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Fast pandas conversion

    # === TEMP STORAGE ===
    .config("spark.local.dir", "/tmp/spark-temp")   # Disk spill location for large shuffles

    # === DEFAULT OPTIONS ===
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", True)
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .getOrCreate()
)

25/10/09 19:53:04 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
#reading in data

tbl_merchants_raw = spark.read.parquet('../data/tables/merchant_data/tbl_merchants.parquet')
consumer_user_details = spark.read.parquet('../data/tables/merchant_data/consumer_user_details.parquet')
transactions21 = spark.read.parquet('../data/tables/transaction_data/transactions_20210228_20210827_snapshot/')
transactions2122 = spark.read.parquet('../data/tables/transaction_data/transactions_20210828_20220227_snapshot/')
transactions22 = spark.read.parquet('../data/tables/transaction_data/transactions_20220228_20220828_snapshot/')
con_fraud_prob = spark.read.option("header","true").csv('../data/tables/merchant_data/consumer_fraud_probability.csv')
merch_fraud_prob = spark.read.option("header", "true").csv('../data/tables/merchant_data/merchant_fraud_probability.csv')

tbl_consumer_raw = spark.read.option("header", "true").csv('../data/tables/merchant_data/tbl_consumer.csv')

transactions = transactions21.unionByName(transactions2122)
transactions = transactions.unionByName(transactions22)

In [5]:
# Load first CSV
postcodes_df = spark.read.csv("../data/income/2024 Locality to 2021 SA2 Coding Index.csv", header=True, inferSchema=True)

# Load second CSV
income_df = spark.read.csv("../data/income/sa2_income.csv", header=True, inferSchema=True)

In [6]:
#Functions
def find_NULL(dfs):

    """Finds any rows with NULLs over different datasets"""

    for df in dfs:
        condition = f.lit(False)
        for col_name in df.columns:
            condition = condition | f.col(col_name).isNull()

        df.filter(condition).show()
    return df.filter(condition).count()

def filter_outliers(data, variables):
    
    """filters outliers of continuous data"""

    n=data.count()
    for feature in variables:
        # Calculate Q1 and Q3
        quantiles = data.approxQuantile(feature, [0.25, 0.75], 0.01)
        q1, q3 = quantiles
        iqr = q3 - q1

        #from ADS lecture slides, n>>100
        scale = m.sqrt(m.log(n)) - 0.5
        if scale<3:
            scale=3
        lower_bound = q1 - scale * iqr
        upper_bound = q3 + scale * iqr
        if lower_bound<0:
            data = data.filter((col(feature) >= 0) & (col(feature) <= upper_bound))
        else:
            data = data.filter((col(feature) >= lower_bound) & (col(feature) <= upper_bound))
    
    return data

def spark_shape(self):
    
    """Easy function for shape of a spark df"""

    return (self.count(), len(self.columns))

pyspark.sql.dataframe.DataFrame.shape = property(spark_shape)

In [7]:
#cleaning tags
string = "name|address|state|postcode|gender|consumer_id"

# Clean consumer table
tbl_consumer = (
    tbl_consumer_raw
    .withColumn("cust_name", f.split(col(string), "\\|").getItem(0))
    .withColumn("address", f.split(col(string), "\\|").getItem(1))
    .withColumn("state", f.split(col(string), "\\|").getItem(2))
    .withColumn("postcode", f.split(col(string), "\\|").getItem(3))
    .withColumn("gender", f.split(col(string), "\\|").getItem(4))
    .withColumn("consumer_id", f.split(col(string), "\\|").getItem(5))
    .drop(string)
)


# Clean merchants table
tbl_merchants = (
    tbl_merchants_raw
    # remove leading (( or [[ and trailing )) or ]]
    .withColumn(
        "tags_clean",
        f.regexp_replace(
            "tags",
            r"^\(\(|^\(\[|^\[\(|^\[\[|\)\)$|\]\)$|\)\]$|\]\]$",
            ""
        )
    )
    # split on `), (` or `], [`
    .withColumn("tags_array", f.split("tags_clean", r"\)\s*,\s*\(|\]\s*,\s*\["))
    # extract each element
    .withColumn("biz_tags", f.lower(f.col("tags_array")[0]))
    .withColumn("rev_band", f.col("tags_array")[1])
    .withColumn("take_rate", f.regexp_extract(f.col("tags_array")[2], r"take rate:\s*([0-9.]+)", 1)
    )
    .drop("tags", "tags_clean", "tags_array")
)

tbl_merchants=tbl_merchants.withColumn("biz_tags", f.regexp_replace("biz_tags", "  ", " "))

In [8]:
#joining transactions and merchants datasets
merchant_transactions=transactions.join(tbl_merchants, on='merchant_abn', how='left')
print(merchant_transactions.shape)
find_NULL([merchant_transactions])

(14195505, 9)


+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
|merchant_abn|user_id|      dollar_value|            order_id|order_datetime|name|biz_tags|rev_band|take_rate|
+------------+-------+------------------+--------------------+--------------+----+--------+--------+---------+
| 29566626791|      8| 74.15732460440282|71a81652-cc91-4bf...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|  18490|107.14809429376949|20149572-a55b-41f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 67202032418|     20| 55.46394975814555|a29071b4-29b3-4f2...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32461318592|     23| 613.9306657410166|4b2e2154-65d8-44f...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 32234779638|     25| 87.15685629102919|e6763664-e95f-4eb...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
| 23633724513|     26|3459.2423030023524|fee9ead7-9ce2-4a4...|    2021-08-20|NULL|    NULL|    NULL|     NULL|
|

580830

In [9]:
merchant_transactions = merchant_transactions.dropna()
print(merchant_transactions.shape)

(13614675, 9)


In [10]:
merchant_transactions.groupBy('name').count().orderBy("count", ascending=True).show()

+--------------------+-----+
|                name|count|
+--------------------+-----+
|Aliquam Eu Institute|    1|
|Consequat Foundation|    1|
|Lobortis Nisi Ass...|    1|
|       Phasellus LLP|    1|
|    Curae Foundation|    1|
|Aenean Gravida In...|    1|
|Elit Dictum Eu Fo...|    1|
|Integer Urna Inst...|    2|
|    Gravida Nunc LLP|    2|
|Adipiscing Fringi...|    2|
|Consectetuer Indu...|    2|
|Accumsan Laoreet ...|    2|
|Semper Pretium Li...|    2|
|    Massa Rutrum LLP|    2|
|            Elit LLP|    2|
|Dictum Mi Corpora...|    2|
|     Sem Corporation|    2|
|Tempor Augue Ac C...|    2|
|  Cras Convallis Ltd|    2|
|        Elit Limited|    2|
+--------------------+-----+
only showing top 20 rows


In [11]:
#filter outliers by biz_tag

n = merchant_transactions.count()
# compute the scale factor
scale = m.sqrt(m.log(n)) - 0.5
stats_by_band = (merchant_transactions.groupby('biz_tags')
                                      .agg(f.expr("percentile_approx(dollar_value, 0.25)").alias("Q1"),
                                           f.expr("percentile_approx(dollar_value, 0.75)").alias("Q3")
                ).withColumn("IQR", f.col("Q3") - f.col("Q1"))
                 .withColumn("lower_bound", f.col("Q1") - scale * f.col("IQR"))
                 .withColumn("upper_bound", f.col("Q3") + scale * f.col("IQR"))
                )
stats_by_band = stats_by_band.drop('IQR')

In [12]:
merchant_transactions = (
    merchant_transactions
    .join(stats_by_band, on="biz_tags", how="left")
    .filter(
        (col("dollar_value") >= col('lower_bound')) &
        (col("dollar_value") <= col("upper_bound"))
    )
    .select(merchant_transactions["*"])
)

merchant_transactions

merchant_abn,user_id,dollar_value,order_id,order_datetime,name,biz_tags,rev_band,take_rate
67609108741,18479,86.4040605836911,d0e180f0-cb06-42a...,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38
47663262928,9,36.69873283148887,c4fcb49a-ce87-4e1...,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66
15299889494,23,69.00314523230432,f7c06643-1b8f-499...,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01
10142254217,18511,70.72395057645588,14b037ad-466d-4e2...,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22
29521780474,18516,68.74975650290615,3c6c21c3-35ae-4d0...,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93
81579058108,42,146.8772319532566,4a0dd7fc-c7d6-4a5...,2021-08-20,Urna Justo Founda...,"cable, satellite,...",a,6.21
31101120643,18533,117.54339257271307,39be6126-7891-44c...,2021-08-20,Commodo Hendrerit...,"cable, satellite,...",a,6.37
72553304202,18536,72.14559015245642,6d7fee0b-a3c8-4c6...,2021-08-20,Rhoncus Proin Nis...,"cable, satellite,...",b,5.04
16587082018,18550,74.46157842663767,515cbef4-673b-488...,2021-08-20,Laoreet Ipsum Ltd,"cable, satellite,...",a,6.29
79645157255,75,17.589399375025362,d7ca9e7a-e492-458...,2021-08-20,Consectetuer Maur...,"cable, satellite,...",a,6.46


In [13]:
print(merchant_transactions.shape)

(13293840, 9)


In [14]:
merchant_transactions=merchant_transactions.withColumnRenamed('name', 'business')
merchant_transactions=merchant_transactions.drop('order_id')
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate
67609108741,18479,86.4040605836911,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38
47663262928,9,36.69873283148887,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66
15299889494,23,69.00314523230432,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01
10142254217,18511,70.72395057645588,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22
29521780474,18516,68.74975650290615,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93
81579058108,42,146.8772319532566,2021-08-20,Urna Justo Founda...,"cable, satellite,...",a,6.21
31101120643,18533,117.54339257271307,2021-08-20,Commodo Hendrerit...,"cable, satellite,...",a,6.37
72553304202,18536,72.14559015245642,2021-08-20,Rhoncus Proin Nis...,"cable, satellite,...",b,5.04
16587082018,18550,74.46157842663767,2021-08-20,Laoreet Ipsum Ltd,"cable, satellite,...",a,6.29
79645157255,75,17.589399375025362,2021-08-20,Consectetuer Maur...,"cable, satellite,...",a,6.46


In [15]:
merchant_transactions.groupBy("biz_tags").agg(
    f.min("dollar_value").alias("min_value"),
    f.max("dollar_value").alias("max_value"),
    f.mean("dollar_value").alias("mean"),
    (f.max("dollar_value") - f.min("dollar_value")).alias("range")
).orderBy("mean", ascending=False).show()

[131.863s][warning][gc,alloc] Executor task launch worker for task 7.0 in stage 76.0 (TID 1277): Retried waiting for GCLocker too often allocating 21270 words
[131.867s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 76.0 (TID 1281): Retried waiting for GCLocker too often allocating 18508 words
[131.871s][warning][gc,alloc] Executor task launch worker for task 4.0 in stage 76.0 (TID 1274): Retried waiting for GCLocker too often allocating 262144 words


25/10/09 19:55:06 WARN TaskMemoryManager: Failed to allocate a page (2097136 bytes), try again.


[132.655s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 76.0 (TID 1272): Retried waiting for GCLocker too often allocating 4872 words
[132.677s][warning][gc,alloc] Executor task launch worker for task 12.0 in stage 76.0 (TID 1282): Retried waiting for GCLocker too often allocating 31293 words


+--------------------+--------------------+------------------+------------------+------------------+
|            biz_tags|           min_value|         max_value|              mean|             range|
+--------------------+--------------------+------------------+------------------+------------------+
|jewelry, watch, c...|   3.409793978681009| 46001.13901942742| 9278.563185654215| 45997.72922544874|
|art dealers and g...|  0.4127496907944707| 10335.94618503865|1966.2357839275696|10335.533435347856|
|             telecom|  0.2931526313090789| 11606.18761084434|1735.6703991984152| 11605.89445821303|
|equipment, tool, ...|0.040595292090802974| 8813.127778854296|1261.6527064827046| 8813.087183562206|
|stationery, offic...|0.004010169952587961| 2333.490127589658| 456.9798017399077| 2333.486117419706|
|health and beauty...| 6.59812930332817E-4|1672.2945523725923| 294.9553375878906| 1672.293892559662|
|motor vehicle sup...|7.092782606876731E-4|1356.3937038188778| 271.4403187831404| 1356.3929

In [16]:
biz_tags_list = merchant_transactions.select("biz_tags").distinct().rdd.flatMap(lambda x: x).collect()
print(biz_tags_list)
print(len(biz_tags_list))

['cable, satellite, and other pay television and radio services', 'computer programming , data processing, and integrated systems design services', 'motor vehicle supplies and new parts', 'furniture, home furnishings and equipment shops, and manufacturers, except appliances', 'bicycle shops - sales and service', 'lawn and garden supply outlets, including nurseries', 'antique shops - sales, repairs, and restoration services', 'books, periodicals, and newspapers', 'digital goods: books, movies, music', 'hobby, toy and game shops', 'opticians, optical goods, and eyeglasses', 'telecom', 'artist supply and craft shops', 'equipment, tool, furniture, and appliance rent al and leasing', 'stationery, office supplies and printing and writing paper', 'health and beauty spas', 'jewelry, watch, clock, and silverware shops', 'music shops - musical instruments, pianos, and sheet music', 'art dealers and galleries', 'florists supplies, nursery stock, and flowers', 'computers, computer peripheral equip

In [17]:
# Count how many distinct biz_tags each business has
biz_tag_counts = (
    merchant_transactions
    .groupBy("business")
    .agg(f.countDistinct("biz_tags").alias("distinct_tag_count"))
    .orderBy(f.desc("distinct_tag_count"))
)



In [18]:
# Assigning the biz_tags to segments
merchant_transactions = merchant_transactions.withColumn(
    "segment",
    f.when(f.col("biz_tags").isin(
        "watch, clock, and jewelry repair shops",
        "jewelry, watch, clock, and silverware shops",
        "shoe shops",
        "antique shops - sales, repairs, and restoration services",
        "gift, card, novelty, and souvenir shops"
    ), "Fashion, Jewelry & Personal Goods")
   
    .when(f.col("biz_tags").isin(
        "books, periodicals, and newspapers",
        "digital goods: books, movies, music",
        "music shops - musical instruments, pianos, and sheet music",
        "art dealers and galleries",
        "artist supply and craft shops",
        "hobby, toy and game shops",
        "cable, satellite, and other pay television and radio services"
    ), "Arts, Media & Entertainment")
   
    .when(f.col("biz_tags").isin(
        "computers, computer peripheral equipment, and software",
        "computer programming , data processing, and integrated systems design services",
        "telecom",
        "equipment, tool, furniture, and appliance rent al and leasing",
        "stationery, office supplies and printing and writing paper"
    ), "Technology & Professional Services")
   
    .when(f.col("biz_tags").isin(
        "furniture, home furnishings and equipment shops, and manufacturers, except appliances",
        "tent and awning shops",
        "lawn and garden supply outlets, including nurseries",
        "florists supplies, nursery stock, and flowers"
    ), "Home, Garden & Living")
   
    .when(f.col("biz_tags").isin(
        "opticians, optical goods, and eyeglasses",
        "health and beauty spas",
        "bicycle shops - sales and service",
        "motor vehicle supplies and new parts"
    ), "Lifestyle, Health & Recreation")
   
    .otherwise("Other")
)
merchant_transactions

merchant_abn,user_id,dollar_value,order_datetime,business,biz_tags,rev_band,take_rate,segment
67609108741,18479,86.4040605836911,2021-08-20,Metus Sit Amet In...,"cable, satellite,...",e,0.38,"Arts, Media & Ent..."
47663262928,9,36.69873283148887,2021-08-20,Eget Lacus LLP,"cable, satellite,...",a,6.66,"Arts, Media & Ent..."
15299889494,23,69.00314523230432,2021-08-20,Porttitor Eros Ltd,"cable, satellite,...",b,5.01,"Arts, Media & Ent..."
10142254217,18511,70.72395057645588,2021-08-20,Arcu Ac Orci Corp...,"cable, satellite,...",b,4.22,"Arts, Media & Ent..."
29521780474,18516,68.74975650290615,2021-08-20,At Sem Corp.,"cable, satellite,...",a,5.93,"Arts, Media & Ent..."
81579058108,42,146.8772319532566,2021-08-20,Urna Justo Founda...,"cable, satellite,...",a,6.21,"Arts, Media & Ent..."
31101120643,18533,117.54339257271307,2021-08-20,Commodo Hendrerit...,"cable, satellite,...",a,6.37,"Arts, Media & Ent..."
72553304202,18536,72.14559015245642,2021-08-20,Rhoncus Proin Nis...,"cable, satellite,...",b,5.04,"Arts, Media & Ent..."
16587082018,18550,74.46157842663767,2021-08-20,Laoreet Ipsum Ltd,"cable, satellite,...",a,6.29,"Arts, Media & Ent..."
79645157255,75,17.589399375025362,2021-08-20,Consectetuer Maur...,"cable, satellite,...",a,6.46,"Arts, Media & Ent..."


In [19]:
merchant_transactions.write.parquet("../data/curated/merchant_transactions", mode="overwrite")

[197.948s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 100.0 (TID 1605): Retried waiting for GCLocker too often allocating 15526 words
[197.964s][warning][gc,alloc] Executor task launch worker for task 6.0 in stage 100.0 (TID 1600): Retried waiting for GCLocker too often allocating 14926 words
[197.977s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 100.0 (TID 1595): Retried waiting for GCLocker too often allocating 13550 words
[197.991s][warning][gc,alloc] Executor task launch worker for task 3.0 in stage 100.0 (TID 1597): Retried waiting for GCLocker too often allocating 1633 words
[198.126s][warning][gc,alloc] Executor task launch worker for task 5.0 in stage 100.0 (TID 1599): Retried waiting for GCLocker too often allocating 54944 words
[198.127s][warning][gc,alloc] Executor task launch worker for task 7.0 in stage 100.0 (TID 1601): Retried waiting for GCLocker too often allocating 13363 words
[198.127s][warning][gc,alloc] Executor t

25/10/09 19:56:13 ERROR Executor: Exception in task 5.0 in stage 100.0 (TID 1599)
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.sql.catalyst.expressions.UnsafeRow.getBinary(UnsafeRow.java:408)
	at org.apache.spark.sql.catalyst.expressions.aggregate.TypedImperativeAggregate.merge(interfaces.scala:589)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator$$anonfun$1.$anonfun$applyOrElse$3(AggregationIterator.scala:201)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator$$anonfun$1.$anonfun$applyOrElse$3$adapted(AggregationIterator.scala:201)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator$$anonfun$1$$Lambda$5176/0x000000012746d518.apply(Unknown Source)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator.$anonfun$generateProcessRow$7(AggregationIterator.scala:215)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator.$anonfun$generateProcessRow$7$adapted(AggregationIterator.scala:209)
	at org.apache.spark.

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
tbl_consumer=tbl_consumer.drop('address', 'cust_name', 'gender')
#tbl_consumer

In [ ]:
mtc_fraud1=merchant_transactions.join(merch_fraud_prob, on=['merchant_abn', 'order_datetime'], how='left')
mtc_fraud1=mtc_fraud1.withColumnRenamed('fraud_probability', 'merch_fraud_prob')
mtc_fraud=mtc_fraud1.join(con_fraud_prob, on=['user_id', 'order_datetime'], how='left')
mtc_fraud=mtc_fraud.withColumnRenamed('fraud_probability', 'con_fraud_prob')
mtc_fraud

In [ ]:
mtc_fraud

In [ ]:
curated=mtc_fraud.groupBy(['merchant_abn', 'user_id']).agg(
    f.count('*').alias('count'),
    f.mean('dollar_value').alias('mean'),
    f.mean('merch_fraud_prob').alias('merch_fraud_prob'),
    f.mean('con_fraud_prob').alias('con_fraud_prob')
)
curated

In [ ]:
new_curated=curated.join(consumer_user_details, on='user_id', how='left')
new_curated=new_curated.join(tbl_consumer, on='consumer_id', how='left')
new_curated=new_curated.drop('consumer_id')
new_curated

In [ ]:
print(new_curated.shape)

In [ ]:
new_curated.write.parquet("data/curated/agg_by_userbiz", mode="overwrite")

In [ ]:
# Rename columns in df2 for easier handling
income_clean = (
    income_df
    .withColumnRenamed("Statistical Areas Level 2 2021 code", "SA2_CODE_2021")
    .withColumnRenamed("Statistical Areas Level 2 2021 name", "SA2_NAME_2021")
)

# Make sure join keys are the same type
postcodes_df = postcodes_df.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`") != 0)

# Perform join on SA2 code
merged_df = postcodes_df.join(income_clean, on="SA2_CODE_2021", how="left")
merged_df = merged_df.drop(income_clean.SA2_NAME_2021)  # drop df2’s version

missing_count = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNull()).count()
print(missing_count)

# Dropping NULL values in income column
print(merged_df.count()) # count before dropping NULL values
merged_df = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNotNull() )
print(merged_df.count()) # count after dropping NULL values

result_df = (
    merged_df
    .groupBy(col("POSTCODE").alias("postcode"))
    .agg(
        f.avg(
            col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`")
        ).alias("median_total_income_2020")
    )
)
result_df

In [ ]:
result_df = result_df.withColumn(
    "income_bin",
    f.when(f.col("median_total_income_2020") < 30000, "<30k")
     .when((f.col("median_total_income_2020") >= 30000) & (f.col("median_total_income_2020") < 40000), "30-40k")
     .when((f.col("median_total_income_2020") >= 40000) & (f.col("median_total_income_2020") < 50000), "40-50k")
     .when((f.col("median_total_income_2020") >= 50000) & (f.col("median_total_income_2020") < 60000), "50-60k")
     .when((f.col("median_total_income_2020") >= 60000) & (f.col("median_total_income_2020") < 70000), "60-70k")
     .when((f.col("median_total_income_2020") >= 70000) & (f.col("median_total_income_2020") < 80000), "70-80k")
     .otherwise("80k+")
)

result_df.show(100, truncate=False)
result_df.printSchema()


In [ ]:
result_df.write.csv("../data/curated/merged_postcode_income.csv", header=True, mode="overwrite")